In [1]:
import ollama
import time
def clear_vram(model_name):
    try:
        ollama.chat(model=model_name, keep_alive=0)
        print("VRAM 정리")
    except Exception as e:
        print(f"언로드 중 오류 발생: {e}")

def get_vram(model_name):
    try:
        ps_info = ollama.ps()
        for m in ps_info.get('models', []):
            if m.get('name') == model_name:
                model_vram_mb = m.get('size_vram', 0) / (1024 ** 2)
                break
    except Exception as e:
        print("VRAM 정보를 가져오는 데 실패했습니다:", e)

In [2]:
#간단한 모델 호출
model_name = 'hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M'

response = ollama.chat(model=model_name, messages=[
    {'role': 'user', 'content': '내 이름은 무엇입니까'},
], options={
    'num_predict': 2000  
})

answer = response['message']['content']

print(answer)

clear_vram(model_name)

나는 Gemma입니다. 😊  무슨 일을 도와드릴까요? 

VRAM 정리


In [1]:
#MK4
from collections import deque
import ollama
import chromadb
import threading
import copy
import time
class ConversationStack_3:
    def __init__(self, max_turns=5):
        self.model_name = 'hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M'

        self.system_prompt = '''
        단답형으로 대답하라, 최대한 간결하게 대답하라
        '''

        self.strategy_information = {
            "lap":15,
            "ranking": 10,
            "pit in": 20,
            "next_tyre" : 'soft'
        }

        self.stack = deque(maxlen=max_turns * 2)
        self.msg_counter = 1
        self.summary_buffer = []
        self.chroma_client = chromadb.PersistentClient(path="./sleipnir_db")
        self.memory_db = self.chroma_client.get_or_create_collection(name="race_history")

    #숏텀 메모리
    def add_message(self, role, content):
        self.stack.append({"role": role, "content": content})
        self.summary_buffer.append({"role": role, "content": content})

        if len(self.summary_buffer) >= 5:
            self.trigger_background_summarization()

    #전략 상황판
    def strategy_board(self, Fixes, Modifications):
        self.strategy_information[Fixes] = Modifications
    
    #인게임 정보
    def telemetry_collection(self):
        #미완성
        self.telemetry
        pass
    
    #요약본 만들기
    def _worker_summarize_and_store(self, messages_to_summarize):
        raw_text = ""
        for msg in messages_to_summarize:
            speaker = "드라이버" if msg['role'] == 'user' else "엔지니어"
            raw_text += f"{speaker}: {msg['content']}\n"

        prompt = "너는 입력에 대한 내용을 요약해 저장하는 장치이다 들어온 입력을 가장 간결하게 줄이고 요약하라 한국어를 사용해"

        try:
            response = ollama.chat(model=self.model_name, messages=[
                {'role': 'system', 'content': prompt},
                {'role': 'user', 'content': raw_text}
                
            ])
            summary_result = response['message']['content']
            
            print(f"\n백그라운드 요약 완료: {summary_result}")

            self.memory_db.add(
                documents=[summary_result],
                metadatas=[{"type": "summary", "timestamp": time.time()}], 
                ids=[f"mem_{time.time()}"] 
            )
            print("DB 저장 완료")
            
        except Exception as e:
            print(f"백그라운드 작업 실패: {e}")

    #롱텀 메모리
    def trigger_background_summarization(self): 
        messages_to_summarize = copy.deepcopy(self.summary_buffer)
        self.summary_buffer.clear()
        threading.Thread(
            target=self._worker_summarize_and_store, 
            args=(messages_to_summarize,), 
            daemon=True
        ).start()

    #에이전트 호출
    def call_sleipnir_with_memory(self, user_input):
        self.add_message("user", user_input)
        messages = [
            {'role': 'system', 'content': self.system_prompt},
            *list(self.stack)
        ]
        print(messages)
        response = ollama.chat(model=self.model_name, messages=messages)
        answer = response['message']['content']
        
        print(f"\n[Sleipnir]: {answer}")
        
        self.add_message("assistant", answer)
    
    #최근 대화 내용 확인
    def check_text(self):
        for msg in self.stack:
            print(f"[{msg['role'].upper()}]: {msg['content']}")

    def check_db_contents(self):
        """
        현재 DB에 저장된 모든 요약 기록을 화면에 출력해서 눈으로 확인합니다.
        """
        # 조건 없이 DB 안의 모든 데이터를 가져옴
        data = self.memory_db.get()
        
        print("\n장기 기억 DB 보관함 확인")
        
        # 문서가 하나도 없는 경우
        if not data['documents']:
            print("현재 DB가 텅 비어있습니다.")
            print("=====================================\n")
            return
            
        # 문서가 있다면 보기 좋게 하나씩 출력
        for i in range(len(data['documents'])):
            doc = data['documents'][i]
            # 저장할 때 넣었던 시간이나 타입 같은 꼬리표(메타데이터)
            meta = data['metadatas'][i] 
            
            print(f"[{i+1}] 텍스트: {doc}")
            print(f"    - 태그: {meta}")
            print("-" * 35)
            
        print(f"총 {len(data['documents'])}개의 요약본이 안전하게 보관되어 있습니다.")
        print("=====================================\n")

    #초기화
    def clear(self):
        self.stack.clear()

agent = ConversationStack_3(5)

In [5]:
a = "나의 이름과 나이는?"     
agent.call_sleipnir_with_memory(a)

[{'role': 'system', 'content': '\n        단답형으로 대답하라, 최대한 간결하게 대답하라\n        '}, {'role': 'user', 'content': '안녕'}, {'role': 'assistant', 'content': '안녕하세요. 😊 \n'}, {'role': 'user', 'content': '나의 이름은 다지게 입니다'}, {'role': 'assistant', 'content': '다지게님, nice to meet you! 👋 \n'}, {'role': 'user', 'content': '나는 12살 입니다'}, {'role': 'assistant', 'content': '알겠어요! 😄  12살이시네요.  \n'}, {'role': 'user', 'content': '나의 이름과 나이는?'}]

[Sleipnir]: 네, 다지게 님 😊  당신은 다지게라고 하셨고 12살입니다. 👍 



In [6]:
agent.check_text()

[USER]: 안녕
[ASSISTANT]: 안녕하세요. 😊 

[USER]: 나의 이름은 다지게 입니다
[ASSISTANT]: 다지게님, nice to meet you! 👋 

[USER]: 나는 12살 입니다
[ASSISTANT]: 알겠어요! 😄  12살이시네요.  

[USER]: 나의 이름과 나이는?
[ASSISTANT]: 네, 다지게 님 😊  당신은 다지게라고 하셨고 12살입니다. 👍 



In [7]:
agent.check_db_contents()


장기 기억 DB 보관함 확인
[1] 텍스트: ## 요약: 

* **드라이버:** 다지게 (12살)
* **엔지니어:** 안녕하세요. 😊 


---

**더 자세한 내용은?**: 

* 드라이버는 자신의 이름을 말합니다.  
* 엔지니어는 친근하게 응답합니다. 

    - 태그: {'timestamp': 1776416075.305098, 'type': 'summary'}
-----------------------------------
총 1개의 요약본이 안전하게 보관되어 있습니다.

